In [ ]:
from pathlib import Path
import cv2 as cvs
import numpy as np
from matplotlib import pyplot as plt

img_path = Path("./test_dots.png")


def find_circle_keypoints(image: np.array):
    gray = cv.cvtColor(image, cv.COLOR_RGB2GRAY)
    _, thesholded = cv.threshold(gray, 127, 255, cv.THRESH_BINARY)
    params = cv.SimpleBlobDetector_Params()
    params.filterByCircularity = True
    params.minCircularity = 0.1
    detector = cv.SimpleBlobDetector_create(params)
    keypoints = detector.detect(thesholded)
    return keypoints


image = cv.imread("../test_dots.png", cv.IMREAD_COLOR)
image = cv.cvtColor(image, cv.COLOR_BGR2RGB)
print(image.shape)

plt.imshow(image)
plt.show()

keypoints = find_circle_keypoints(image)

blank = np.zeros((1, 1))
blobs = cv.drawKeypoints(
    image, keypoints, blank, (255, 0, 0), cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)


plt.imshow(blobs, interpolation="nearest")
plt.show()

[ WARN:0@0.294] global loadsave.cpp:275 findDecoder imread_('../test_dots.png'): can't open/read file: check file path/integrity


error: OpenCV(4.12.0) /Users/xperience/GHA-Actions-OpenCV/_work/opencv-python/opencv-python/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'


In [ ]:
from pathlib import Path
import cv2 as cv
import numpy as np
from matplotlib import pyplot as plt

img_path = Path("./test_dots.png")

image = cv.imread("../test_dots.png", cv.IMREAD_COLOR_RGB)
print("test_dots.png")
plt.imshow(image)
plt.show()


image = cv.medianBlur(image, 5)
gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
print("Gray Scale")
plt.imshow(gray)
plt.show()

_, thesholded = cv.threshold(gray, 128, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
print("Thresholded")
plt.imshow(thesholded)
plt.show()

params = cv.SimpleBlobDetector_Params()
params.filterByCircularity = True
params.minCircularity = 0.1
detector = cv.SimpleBlobDetector_create(params)
keypoints = detector.detect(thesholded)

blank = np.zeros((1, 1))
blobs = cv.drawKeypoints(
    image, keypoints, blank, (255, 0, 0), cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

print("Blobs detected")
plt.imshow(blobs, interpolation="nearest")
plt.show()

In [ ]:
for kp in keypoints:
    print(kp.pt)

In [ ]:
from dataclasses import dataclass
from math import hypot


@dataclass
class Point:
    x: float = 0.0
    y: float = 0.0

    @classmethod
    def from_tuple(cls, coords):
        return cls(coords[0], coords[1])

    def as_tuple(self):
        return (self.x, self.y)


def euc_dist(a: Point, b: Point) -> float:
    return hypot(b.x - a.x, b.y - a.y)


points = [Point.from_tuple(kp.pt) for kp in keypoints]

print(f"There are {len(points)} points in the image.")

In [ ]:
# lets test out the finding rectangles stuff
from itertools import combinations


# util function used to put the distances into buckets
def bucket(value: float, size: float):
    return int(value / size)


# compute all the unordered pairs
pairs = list(combinations(points, 2))

print(f"There are {len(pairs)} pairs of points.")

In [ ]:
# work out the distances between each pair
bucketed = [bucket(euc_dist(a, b), 5) for a, b in pairs]

print(f"There are {len(list(set(bucketed)))} pairs of different distances.")

In [ ]:
# find the lines with approx the same lengths
from collections import defaultdict


tally = defaultdict(list)
for pair, dist in zip(pairs, bucketed):
    tally[dist].append(pair)
same_distance_pairs = dict(
    ((dist, pairs) for dist, pairs in tally.items() if len(pairs) > 1)
)
same_distance_pairs

In [ ]:
# each of the pairs of pairs for each bucket are possibly rectangles
# so we compute the pairs of pairs and test if they have the 'same'
# centre points


def same_midpoint(pairA, pairB, threshold=5):
    # note - threshold is in pixels
    def midpoint(pair):
        a, b = pair
        return Point((b.x - a.x) / 2.0, (b.y - a.y) / 2.0)

    # compute the midpoints of each pair
    midA = midpoint(pairA)
    midB = midpoint(pairB)

    # are they the same? or rather are they some delta distant
    distance = euc_dist(midA, midB)

    return distance < threshold


all_rects = []
for _, pairs in same_distance_pairs.items():
    pairs_of_pairs = combinations(pairs, 2)
    rects = [(p1, p2) for (p1, p2) in pairs_of_pairs if same_midpoint(p1, p2)]
    all_rects.extend(rects)

for rect in all_rects:
    print(rect)
    print()

In [ ]:
out = image.copy()

for rect in all_rects:
    p0, p2 = rect[0]
    p1, p3 = rect[1]

    points = np.array(
        [[p0.x, p0.y], [p1.x, p1.y], [p2.x, p2.y], [p3.x, p3.y]], np.int32
    )

    points = points.reshape((-1, 1, 2))

    cv.polylines(out, [points], True, (255, 0, 0))

plt.imshow(out)
plt.show()

In [ ]:
out = image.copy()


def point_as_ints(point):
    return int(point[0]), int(point[1])


for rect in all_rects:
    p0, p2 = rect[0]
    p1, p3 = rect[1]

    p0, p2, p1, p3 = p0.as_tuple(), p2.as_tuple(), p1.as_tuple(), p3.as_tuple()
    p0, p2, p1, p3 = [point_as_ints(p) for p in [p0, p2, p1, p3]]

    cv.line(out, p0, p2, (255, 0, 0), 10)
    cv.line(out, p1, p3, (255, 0, 0), 10)

plt.imshow(out)
plt.show()

Now we have an issue with some of the rectangles being the same but with the different lines. So now we want to identify rectangles that contain the same points and then pick one are the correct one.

It doesn't matter which so long as we do it deterministically.

Turns out we want to find a convex hull around the points and there is an OpenCV function for that.

In [ ]:
# turn each rect into a set of points
def to_point_set(rect):
    p1, p2 = rect[0]
    p3, p4 = rect[1]
    return frozenset([p1.as_tuple(), p2.as_tuple(), p3.as_tuple(), p4.as_tuple()])


# remove duplicate rects
rects_no_dups = {to_point_set(s) for s in all_rects}

rects_no_dups

In [ ]:
# now translate them into convex hulls
def compute_hull(rect):
    points = np.array(list(rect), dtype=np.int32)
    hull = cv.convexHull(points, clockwise=False)
    return hull


hulls = [compute_hull(rect) for rect in rects_no_dups]

In [ ]:
out = image.copy()

for hull in hulls:
    cv.polylines(out, [hull.reshape(-1, 1, 2)], True, (255, 0, 0), 10)

plt.imshow(out)
plt.show()

Ok now we need to come up with a way to decode the id's that are encoded in the dots.

To do that we need a cononical way to start with the red dot and go CW or CCW around. Either is fine, we just have to pick one.

We can find the red dot by extracting the mean colour value for the pixels under the blob (we will use some radius around the centroid as proxy). Then finding the closest colour in our encoding colours in rgb colour space (or something - need to research colour distances). Or you don't because the colour is to far away and then it's a decoding error.

In [ ]:
hulls[0]

In [ ]:
# Find the colours for all the dots
num_points = len(hulls) * 4  # there are 4 points per hull - they are rectangles
point_colours = np.zeros((num_points, 3), dtype=np.uint8)  # 3 components per colour
for hull_idx, hull in enumerate(hulls):
    for point_idx, point in enumerate(hull):
        x, y = point[0]
        colour = image[y, x]
        point_arr_idx = hull_idx * 4 + point_idx
        point_colours[point_arr_idx] = colour

point_colours

In [ ]:
# copy the pallette in - this will need to be the same for the page generation code
palette = np.array(
    [
        [255, 95, 0],  # Electric Blue  (#005FFF)
        [192, 35, 106],  # Deep Indigo    (#6A23C0)
        [215, 255, 79],  # Mint Green     (#4FFFD7)
        [252, 132, 192],  # Lavender       (#C084FC)
        [0, 255, 246],  # Acid Yellow    (#F6FF00)
        [100, 79, 255],  # Fluoro Coral   (#FF4F64)
        [0, 122, 255],  # Sunset Orange. (#FF7A00)
        [200, 0, 255],  # Magenta Pink   (#FF00C8)
        [150, 200, 0],  # Teal Green     (#00C896)
        [0, 180, 255],  # Warm Mustard   (#FFB400)
    ],
    dtype=np.uint8,
)


def nearest_colour_neighbours(colour_a, colour_b):
    """Computes the nearest neighours for an array of colours

    Args:
        colour_a (np.array): (N, 3) a row for each color and 3 components
        colour_b (np.array): (M, 3) a row for each color and 3 components
    """
    # we are just going to use euc distance but could use other colour spaces
    dists = np.linalg.norm(colour_a[:, None, :] - colour_b[None, :, :], axis=2)  # (N,M)
    nearest_indices = dists.argmin(axis=1)
    return nearest_indices


nearest_colour_neighbours(point_colours, palette)